# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type:** Ranking / scoring, built on top of a classification model.

"Which pages should be reviewed first" is literally a ranking question — the
framing-ml-problems mapping is "which ones first? -> ranking -> precision@K."
But the model underneath is a binary classifier (declining vs not), and I turn
its predicted probability into a rank by sorting. So the honest framing is:
classification model, ranking use-case, ranking metric. I'm naming this
explicitly so I don't end up reporting accuracy later and calling it done —
accuracy isn't the real success criterion here, precision@K is.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (for this notebook, starter data):** is_declining_label =
(trend_direction == "down")

**Where it comes from:** a defined proxy, not an observed outcome.
trend_direction is computed from trend_pct within the same 90-day window I'd
be scoring, so it describes the current state, not a future event.

**Why I'm using it anyway, for now:** it lets me build and validate the
ranking mechanics (baseline -> model -> precision@K) on a labeled dataset I
already have, before adding the warehouse's time-window complexity.

**The real target, once I move to the warehouse release (data-contract
week):** prior_90d_features -> decline in a later, non-overlapping 30-day
window. An observed outcome, not a same-window bucket -- this is the
leakage fix Notebook 02 already demonstrated the need for.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns")

30,000 rows, 44 columns


In [3]:
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))
df[["content_id", "trend_direction", "trend_pct", "is_declining_label"]].head(5)

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


,content_id,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,down,-41.4,1
1,content_a1fb4e703a9e,down,-57.7,1
2,content_9aa793d4d895,down,-60.9,1
3,content_331d6c4de07b,stable,-13.8,0
4,content_d99b7a2d90ca,down,-34.7,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**, with the base rate always printed alongside it.

Review capacity is the real constraint (an editor can only act on so many
pages), so precision@K matches how the output actually gets used -- not
generic accuracy. I'm picking K=50 as primary because that's the queue depth
the starter pipeline already validates against, but I'll report Precision@20
too since Notebook 02 showed the two can disagree (a rule can win at the very
top of the list while a model wins deeper in).

"Good" means: precision@50 clearly above the base rate (0.542 -- just over
half of pages are labeled declining by the proxy, so a naive/random ranking
already gets ~0.54 for free). Anything close to that is not a finding.

* Action supported: Every week, the system ranks pages by predicted decline risk. Editors review the top 50 pages first to decide which content should be refreshed before performance drops further.

In [5]:
base_rate = df["is_declining_label"].mean()
print(f"Base rate (declining proxy): {base_rate:.3f}")
print("Any ranking must clear this to mean anything.")

Base rate (declining proxy): 0.542
Any ranking must clear this to mean anything.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (content_id)**, aggregated over a trailing
90-day window in the starter data (a future warehouse-built window once I
move to the capstone data).

In [7]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label'],
      dtype='object')

In [6]:
unit_cols = ["content_id", "client_id", "impressions_90d",
             "days_since_last_update", "avg_position", "ctr",
             "word_count", "trend_direction", "is_declining_label"]
print(f"{df['content_id'].nunique():,} unique pages across {df['client_id'].nunique()} clients")
df[unit_cols].head(5)

30,000 unique pages across 32 clients


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Notebook 01's full pipeline already shows the gap on this exact data: the
hand-written baseline rule (stale + visible) scores Precision@50 = 0.240,
while the random forest -- trained with a client-holdout split, so no
client's pages leak between train and test -- scores Precision@50 = 0.740,
roughly 3.1x the rule.

That gap is the case for ML: a single if-statement (stale AND visible) can't
weigh six-plus signals (impressions, position, CTR, age, freshness, word
count) against each other or capture interactions between them, especially
deeper in a ranked list where the obvious cases run out. This is a *directional*
result from a 30k-row anonymized slice, not a benchmark on the full 79M-row
warehouse -- that comparison has to be re-earned there with proper validation.

In [11]:
import subprocess
result = subprocess.run([sys.executable, "scripts/run_all.py"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

In [12]:
import json
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {base:.3f}")
print(f"Random forest Precision@50: {rf:.3f}  ({rf/base:.1f}x the rule)")
print(f"Split strategy: {res['split_strategy']}")

Baseline rule Precision@50: 0.240
Random forest Precision@50: 0.740  (3.1x the rule)
Split strategy: client_holdout


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.